# Teste isolado — EPE (Empresa de Pesquisa Energética)

Fonte candidata: EPE, setor Energia (federal, vinculada ao MME). Notebook
**descartável** (Fase 1) — sem dispatcher, sem gravar nada.

## Confirmado antes de assumir

**Não confirmado ainda, ao contrário do padrão de outras fontes gov.br**:
não sei se é a mesma plataforma Plone do ANTT/ANEEL — a URL não segue o
padrão `.../2026-defeso-eleitoral/{slug}` que essas duas têm. Pode ser
outro CMS, mesmo sendo portal padrão `.gov.br`.

**Achado 1**: uma tentativa de fetch direto trouxe conteúdo de 2023,
claramente desatualizado (o site tem matéria de 03/08/2026 confirmada por
busca externa) — indício de cache agressivo em algum ponto do caminho.
**Testa com `httpx`/`curl_cffi` direto aqui, sem confiar em cache de
terceiros.**

**Achado 2**: existe paginação por sufixo tipo `/area-1` (visto num
resultado de busca), possivelmente ligada ao filtro "Área de Interesse" da
página — não confirmado se é paginação simples ou filtro de categoria.

Esse notebook é exploratório: a Etapa 1 primeiro **descobre a estrutura
real** (imprime HTML bruto de um trecho, testa padrões de link) antes de
qualquer parser assumir seletor específico.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml feedparser
dbutils.library.restartPython()


In [0]:
import re
import time
import random
import urllib.parse
from typing import Optional

import httpx
import feedparser
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
SITE_URL = "https://www.epe.gov.br/pt/imprensa/noticias"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)


In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer="https://www.epe.gov.br/")
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


## Etapa 0 — Confirmar que NÃO estamos pegando cache velho

Checa se a data mais recente que aparece na página bate com algo próximo
de hoje (não 2023/2024).

In [0]:
html = baixar_pagina(SITE_URL)
if not html:
    print("Falha ao baixar a página.")
else:
    print(f"HTML baixado: {len(html)} chars\n")

    datas_encontradas = re.findall(r"\d{2}/\d{2}/(\d{4})", html)
    anos_distintos = sorted(set(datas_encontradas), reverse=True)
    print(f"Anos encontrados no HTML (via regex DD/MM/AAAA): {anos_distintos}")
    print("Se o ano mais recente for 2025/2026, o conteúdo está fresco -- prossiga.")
    print("Se for só 2023 ou anterior, é cache velho -- não confie no restante do teste.")


## Etapa 1 — Descobrir a estrutura real da listagem

Não assume seletor nenhum ainda. Extrai todo `<a href>` que aparente ser
notícia (`/pt/imprensa/noticias/{slug}`, não a página raiz nem filtro), e
imprime o HTML ao redor do primeiro item pra inspeção manual.

In [0]:
soup = BeautifulSoup(html, "lxml")

candidatos = []
vistos = set()
for tag_a in soup.find_all("a", href=True):
    href = tag_a["href"].strip()
    url_absoluta = urllib.parse.urljoin(SITE_URL, href)

    if not url_absoluta.startswith("https://www.epe.gov.br/pt/imprensa/noticias/"):
        continue
    if url_absoluta == SITE_URL or url_absoluta in vistos:
        continue
    vistos.add(url_absoluta)

    candidatos.append({"texto_ancora": tag_a.get_text(" ", strip=True), "url": url_absoluta,
                        "tag_pai": tag_a.parent.name if tag_a.parent else None})

print(f"{len(candidatos)} links candidatos a notícia encontrados.\n")
for c in candidatos[:15]:
    print(f"[{c['tag_pai']}] {c['texto_ancora'][:70]!r}")
    print(f"  -> {c['url']}")


## Etapa 2 — Inspecionar o HTML bruto ao redor do primeiro item

Sem assumir classe/seletor — só imprime o trecho, pra decidir o parser real
depois de ver a estrutura de verdade (evita repetir o erro do fetch com
cache: aqui é output direto da nossa própria requisição).

In [0]:
if candidatos:
    primeiro_link = soup.find("a", href=lambda h: h and candidatos[0]["url"].endswith(h.rstrip("/").split("/")[-1]))
    if primeiro_link:
        contexto = primeiro_link.find_parent(["article", "div", "li"]) or primeiro_link.parent
        print("HTML bruto ao redor do primeiro item candidato:\n")
        print(str(contexto)[:2000])
    else:
        print("Não achei o elemento pai do primeiro link -- inspecione soup diretamente.")


## Etapa 3 — Testar se existe feed RSS (padrão gov.br costuma ter)

In [0]:
for candidato_feed in [
    "https://www.epe.gov.br/pt/imprensa/noticias/RSS",
    "https://www.epe.gov.br/pt/imprensa/noticias?rss=1",
    "https://www.epe.gov.br/feed",
]:
    resp = httpx.get(candidato_feed, headers=headers_aleatorios(), timeout=15, follow_redirects=True)
    content_type = resp.headers.get("content-type", "")
    print(f"{candidato_feed} -> status={resp.status_code}, content-type={content_type!r}")
    if "xml" in content_type.lower() or "rss" in content_type.lower():
        parsed = feedparser.parse(resp.content)
        print(f"  -> parece RSS válido! {len(parsed.entries)} entries.")


## Conclusão da Fase 1

Preencher depois de rodar: a Etapa 0 confirma se o conteúdo é fresco; a
Etapa 1/2 mostram a estrutura real da listagem (sem suposição); a Etapa 3
confirma se existe atalho de RSS. Só depois desse diagnóstico decide entre
Dispatcher 1 (se achar RSS) ou Dispatcher 3 (scraping, com seletor real
identificado na Etapa 2 — não herdado às cegas do ANTT/ANEEL, já que a
plataforma pode ser diferente).